# 정답 노트북

이 파일은 빈칸을 채운 학습용 정답입니다. 먼저 빈칸 노트북으로 직접 풀어 본 뒤 비교하세요.


* 학년: 
* 반:
* 번호:
* 이름:


> 이 파일은 **정답(코드 채움) 버전**입니다. 빈칸 연습은 같은 폴더의 원래 노트북을 사용하세요.


# 실습 4: 쇼핑몰 고객 군집 모델 구현하기

연 소득과 소비 점수가 비슷한 고객을 k-평균 군집화로 묽고, 실루엣 점수로 군집 품질을 평가해 보자.


## 단계 0: 준비 (라이브러리 설치)


In [ ]:
# 라이브러리는 JupyterLite 커널에 미리 포함되어 있습니다.
# (Colab에서는 아래 설치 코드가 실행됩니다.)
import sys

if sys.platform != "emscripten":
    import importlib, subprocess
    for module_name, pip_name in [
        ("pandas", "pandas"),
        ("numpy", "numpy"),
        ("matplotlib", "matplotlib"),
        ("seaborn", "seaborn"),
        ("sklearn", "scikit-learn"),
    ]:
        try:
            importlib.import_module(module_name)
        except ModuleNotFoundError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=False)

from sklearn import set_config
set_config(display="text")

print("라이브러리 준비 완료")


In [ ]:
# pandas를 가져오고 Mall_Customers.csv를 df로 불러오기
import pandas as pd
df = pd.read_csv('Mall_Customers.csv')
print(df.head())
print(df.info())
print(df.isnull().sum())


In [ ]:
# 연 소득과 소비 점수 열만 선택해 data에 저장하고 처음 5행 확인하기
data = df[['Annual Income (k$)', 'Spending Score (1-100)']]
data.head()


In [ ]:
# KMeans를 가져와 군집 5개인 model 만들기(random_state=42, n_init=10)
from sklearn.cluster import KMeans
model = KMeans(n_clusters=5, random_state=42, n_init=10)
model.fit(data)


In [ ]:
# model.labels_를 df의 cluster 열에 저장하기
df['cluster'] = model.labels_
centers = model.cluster_centers_
print(centers)
df[['Annual Income (k$)', 'Spending Score (1-100)', 'cluster']].head(10)


In [ ]:
# 연 소득-소비 점수 산점도를 cluster별 색으로 표시하기
import matplotlib.pyplot as plt
import seaborn as sns
sns.scatterplot(
    data=df,
    x='Annual Income (k$)',
    y='Spending Score (1-100)',
    hue='cluster',
    palette='Set2'
)
plt.scatter(centers[:, 0], centers[:, 1], c='black', marker='*', s=200)
plt.show()


In [ ]:
# silhouette_score를 가져와 data와 model.labels_로 실루엣 점수 구하기
from sklearn.metrics import silhouette_score
silhouette_score(data, model.labels_)


In [ ]:
# ks=range(2,9), scores=[] 준비하기
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
ks = range(2, 9)
scores = []
for k in ks:
    m = KMeans(n_clusters=k, random_state=42, n_init=10)
    m.fit(data)
    scores.append(silhouette_score(data, m.labels_))
for k, s in zip(ks, scores):
    print(k, s)
print('최적 k:', list(ks)[scores.index(max(scores))])
